In [1]:
from pathlib import Path

import util
from util import workflow

browser = True
file = util.notebook_file() if util.is_notebook() else __file__
tag = util.file_tag(file)
root_path = Path("..")
data_path = util.data_path(root_path)

In [2]:
# Build
from automol.graph import enum

import automech
from automech.species import Species

mech0 = automech.io.read(data_path / "full_raw.json")
mech = automech.from_smiles(spc_smis=["C1=CCCC1", "C12C(O2)CCC1"], src_mech=mech0)
#  - add HO2 addition to *ene*
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.pi2_addition,
    rcts_=[None, "O[O]"],
    spc_col_=Species.smiles,
    src_mech=mech0,
)
#  - add ring-forming scission
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.ring_forming_scission,
    src_mech=mech0,
)
#  - enumerate HO2 abstractions from *ene* and *1-2epoxy*
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.abstraction,
    rcts_=[["C1=CCCC1", "C12C(O2)CCC1"], "O[O]"],
    spc_col_=Species.smiles,
    src_mech=mech0,
)

  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# Write
workflow.write(mech=mech, tag=tag, root_path=root_path, browser=browser)


Finalizing build for...
reactions=shape: (8, 5)
┌────────────────┬────────────────────────────┬───────────┬────────────┬───────────────────────────┐
│ reactants      ┆ products                   ┆ formula   ┆ reversible ┆ rate                      │
│ ---            ┆ ---                        ┆ ---       ┆ ---        ┆ ---                       │
│ list[str]      ┆ list[str]                  ┆ struct[3] ┆ bool       ┆ struct[17]                │
╞════════════════╪════════════════════════════╪═══════════╪════════════╪═══════════════════════════╡
│ ["S(722)"]     ┆ ["C5H8(522)", "HO2(8)"]    ┆ {9,2,5}   ┆ true       ┆ {1,{null,null,null,null,n │
│                ┆                            ┆           ┆            ┆ ull,n…                    │
│ ["S(722)"]     ┆ ["C5H8O(825)", "OH(4)"]    ┆ {9,2,5}   ┆ true       ┆ {1,{null,null,null,null,n │
│                ┆                            ┆           ┆            ┆ ull,n…                    │
│ ["C5H7(1202)", ┆ ["C5H8(522)", "HO2(8)"]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]


Writing mechanism...
../data/B_rh-ho2_p1v1_raw.json
../data/B_rh-ho2_p1v1.json
../data/mechanalyzer/B_rh-ho2_p1v1.dat
../data/mechanalyzer/B_rh-ho2_p1v1.csv


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]


Stereoexpansion errors:


In [4]:
# Read
workflow.read(tag=tag, root_path=root_path)


Reading mechanisms...

Adding calculated thermo...

Adding calculated rates...

Expanding and updating parent...


  0%|          | 0/3151 [00:00<?, ?it/s]


Writing mechanism...
../data/B_rh-ho2_p1v1_calc.json
../data/full_B_rh-ho2_p1v1_control.json
../data/full_B_rh-ho2_p1v1_calc.json
../data/chemkin/full_B_rh-ho2_p1v1_control.dat
../data/chemkin/full_B_rh-ho2_p1v1_calc.dat

Compare calculated mechanism to parent mechanism...

1. Original (compare):
S(722) = C5H8O(825) + OH(4)                              1.000      0.000      0.000
S(722) = C5H8(522) + HO2(8)                              1.000      0.000      0.000
C5H8O(825) + HO2(8) = C5H7O(758) + H2O2(10)              1.000      0.000      0.000
C5H8(522) + HO2(8) = C5H7(504) + H2O2(10)                1.000      0.000      0.000
C5H8(522) + HO2(8) = C5H7(500) + H2O2(10)                1.000      0.000      0.000
C5H8(522) + HO2(8) = C5H8O(825) + OH(4)                  1.000      0.000      0.000

2. Calculated (compare):
C5H7(504) + H2O2(10) = C5H8(522) + HO2(8)                     1.000      0.000      0.000
C5H7O(758)rs0 + H2O2(10) = C5H8O(825)rs + HO2(8)              1.000      0.

In [5]:
# # Simulate
# workflow.simulate(full_tag=f"full_{tag}_calc", root_path=root_path)
# workflow.simulate(full_tag=f"full_{tag}_control", root_path=root_path)